In [0]:
import importlib
import lib.checker_handler as cd
import lib.checker as chk
from lib.utils import parse_yaml, find_job

#Structure

- **Table Metadata Layer**: Contains information about the tables that will be checked for quality, including:
  - Table Name:
  - Table Schema
  - Table Format
  - Path to the table metadata file, inclusing schema and tests
  - Dataframe
- **Column Metadata Layer**: Contains information about a tabke's columns, such as:
  - Column Name
  - Column Type
  - Nullable
  - Mandate: Is it mandatory? Is it required? Is it optional?
  - date_predicate: is it a watermark column?
  - key: is it a key for this data set?
  - tests: Contains the tests to be poerformed against the column
    - test_type: duplication, missing, type_missmatch...
    - kwargs: Can have the arguments to perform the test. Such as thresholds or regex expressions

![Sample column metadata](misc/Example table.png)
- **Checker**: Handles the individual processing of tables, running over all the tests assigned for each table and storing them into a dataframe
- **Checker handler**: Logical piece tha manages the information received from the table metadata layer, and instructs the checker to perform the validations. Aggregates the individual results obtained and saves it, both at the analytical level (current snapshot) but also at the aggregate level (for hitorical reporting).

In [0]:
"""
The handler requires a specific format of TAble metadata coming in, which is a little diffrent than what I have set up 
in my jobs definitions. So in here I'm just adapting the info there to be able to consume it.
"""

table_meta = find_job('silver' , 'incremental', 'silver_incremental')
tables = table_meta.get('tables')
instructions = []
for table in tables:
    table['dataframe']= spark.sql(f"SELECT * FROM {table.get('catalog')}.{table.get('schema')}.{table.get('table_name')}")
    table['metadata_path'] = f"{table.get('catalog')}/{table.get('table_name')}/{table.get('table_name')}.yml"
    instructions.append(table)

prepped = {
    entry['table_name']: {k:v for k, v in entry.items()}
    for entry in instructions
}    
prepped

In [0]:
handler = cd.CheckerHandler(spark, prepped)
x = handler.execute()